ML FINAL EXAM
Email:
Task: Copy this notebook on your drive and answer in that copy

Choose a dataset of your choice from kaggle or UCI

Some suggestions:https://www.kaggle.com/datasets/ahmettezcantekin/beginner-datasets

You may choose a dataset of your choice too

In this exam:

Provide code and explaination(in text cell) whenever needed and you must show the outputs
Before submitting run all cells and make sure the outputs are visible

0. Dataset overview
Why you choose this dataset and what did you observe from the dataset description

Answer:
I selected the Titanic dataset. The dataset contains information about passengers who were on the Titanic, including their passenger class, age, gender, ticket information, fare, and survival status.

I chose this dataset because it is a well-known dataset for learning machine learning and data analysis. As i already use this dataset many times before, so i feel this dataset provide a good opportunity to data exploration, preprocessing, feature selection and train-test spliting.

1. Dataset description (15 marks)
Dataset Description
How many features?
Classification or regression problem? Why do you think so?
How many data points?
Is there any null values?
What kind of features are in your dataset? (Quantitative / Categorical)
Do you need to encode the categorical variables, why or why not?
Correlation of all the features, What do you understand after the correlation test?
Perform exploratory data analysis to extract some important relationships from your data.
Provide necessary codes and explanation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dataset_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(dataset_url)

df.head()

In [ ]:
print("Total columns:", df.shape[1])
print("Number of input features:", df.shape[1] - 1)
print("Target variable:", "Survived")

The dataset contains 12 coloumns. Survived is the target variable and others are features. So, 11 features and 1 target variable in this dataset.

In [ ]:
print("Unique values in Survived:", df["Survived"].unique())

print("\nNumber of classes:", df["Survived"].nunique())

This is a classification problem. Because target variable has 2 values 0 and 1. 0 is for non-survival and 1is for survival.

In [ ]:
print("Total data points:", df.shape[0])

Total data points is 891.

In [ ]:
df.isnull().sum()

Yes, there are null values in the dataset.

In [ ]:
print("Quantitative features:")
print(df.select_dtypes(include=["int64", "float64"]).columns.tolist())

print("\nCategorical features:")
print(df.select_dtypes(include=["object"]).columns.tolist())

The dataset contains both quantitative and categorical features. passengerId, survived, Pclass, age, sibsp, parch ,fare are Quatitative features and Name, sex, ticket, cabin, embarked are categorical features.

Yes, categorical variables need to be encoded. Because, most of the Machine learning algorithms need to be cateogrized before train the model.

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Heatmap")
plt.show()

Pclass has a strong negative correlation with survival, while Fare shows a positive correlation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='Survived', hue='Sex', ax=axes[0])
axes[0].set_title("Survival Count by Sex")

sns.boxplot(data=df, x='Pclass', y='Age', ax=axes[1])
axes[1].set_title("Age Distribution by Pclass")
plt.tight_layout()
plt.show()

Females had a significantly higher survival rate than males. First-class passengers were generally older and had higher survival rates.

2. Dataset pre-processing (15 marks)
Provide code
Discuss the pre processing steps you applied and why?

In [ ]:
# Drop 'Cabin' (too many missing values) and uninformative identifier columns
df_clean = df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'])

# Impute missing values
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Categorical Feature Encoding
df_clean = pd.get_dummies(
    df_clean,
    columns=['Sex', 'Embarked'],
    drop_first=True,
    dtype=int
)

# Display the cleaned dataset information
print("Cleaned Dataset Shape:", df_clean.shape)

df_clean.head()

Cabin was dropped due to >75% missing data. Uninformative unique identifiers (PassengerId, Name, Ticket) were removed to prevent overfitting.
Age was filled using the median to handle missing values without being skewed by extreme values. Embarked was filled using the mode.
One-Hot Encoding (pd.get_dummies) was applied to categorical features (Sex and Embarked) to convert them into numeric vectors.

3.Feature selection and Dataset splitting (10 marks)
Which features you wanna keep ? Justify and drop and rest or apply any other feature engineering step
Perform Train test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Feature Selection
X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Survived was selected as the target variable y, while all cleaned attributes were selected as feature inputs X. I divided the dataset into 80% training data and 20% testing data. The training data will be used to train the machine learning model, while the testing data will be used to evaluate the model on unseen data. I used random_state=42 to make the split reproducible and stratify=y to maintain a similar proportion of survived and non-survived passengers in both the training and testing datasets.

4. Pipeline Creation (Supervised) (10 marks)
Select 2 models of your choice and build 2 pipelines for them

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Models creation:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Pipeline 1
logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', models['Logistic Regression'])
])

# Pipeline 2
random_forest_pipeline = Pipeline([
    ('model', models['Random Forest'])
])

5. Model Training (5 marks)
Train those 2 models

In [ ]:
logistic_pipeline.fit(X_train, y_train)

random_forest_pipeline.fit(X_train, y_train)

6. Model selection/Comparison analysis (15 marks)
Bar chart showcasing prediction accuracy of all models (for classification)
Precision, recall comparison of each model. (for classification)
Confusion Matrix (for classification)
R2 score and Loss (for regression)
Compare the results of all models based on all of the above described metrics. Why do you think this model performed better than the other one for this dataset?

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

results = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        'Model': name, 
        'Accuracy': acc, 
        'Precision': prec, 
        'Recall': rec, 
        'F1-Score': f1
    })
    
    print(f"\n--- {name} Confusion Matrix ---")
    print(confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results)

print("\n--- Model Comparison Summary ---")
print(results_df.round(4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(results_df['Model'], results_df['Accuracy'])

plt.title('Accuracy Comparison of Two Models')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

plt.show()

Logistic Regression performed better overall than Random Forest on the Titanic dataset. Logistic Regression achieved an accuracy of 80.45%, while Random Forest achieved an accuracy of 65.92%. Logistic Regression also achieved higher precision (79.31%) and F1-score (72.44%) compared to Random Forest, which achieved 53.85% precision and 64.74% F1-score. Overall, Logistic Regression performed better because it produced a better balance between accuracy, precision, and recall for this dataset.

7. Treating the problem as Unsupervised (20 marks) ( Explore the topic as you wish )
Treat the problem as a unsupervised problem and perform any unsupervised model and evalute the result
Which method worked better? supervised or unsupervised approach and why?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Apply K-Means Clustering 
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_train_scaled)

print("Silhouette Score:", silhouette_score(X_train_scaled, clusters))
print("Adjusted Rand Index (ARI):", adjusted_rand_score(y_train, clusters))

The supervised learning approach worked better than the unsupervised approach for this Titanic dataset. In supervised model, Logistic Regression, achieved an accuracy of 80.45%, with a precision of 79.31%, recall of 66.67%, and F1-score of 72.44%. In unsupervised K-Means approach, the Silhouette Score was 0.2375 and the Adjusted Rand Index (ARI) was 0.1304. So, we can say that, Supervised learning approach performed better.

8. Self Reflection on this machine learning course (10 marks)
Explain the hardest and the easiest topic of this course according to you in a intuitive way (you may also provide real world implementation , necessity etc along with the explaination)

Answer:
Hardest Topic: For me, the hardest topic was feature engineering and model evaluation. I found it challenging to decide which features were useful and which features should be removed.

Easiest Topic: For me, data preprocessing was the easiest topic to understand. I found this topic easier because the steps are straightforward and easy to apply to different datasets. It also helped me understand how raw data can be prepared for machine learning.